# Mask R-CNN - Train on JuniperMapper

This notebook shows how to train Mask R-CNN on [JuniperMapper dataset](https://zenodo.org/records/10058715).

In [ ]:
from matplotlib import pyplot as plt
import os
import sys
import numpy as np

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'bin'))
import mrcnn.model as modellib
from mrcnn import utils # type: ignore
from mrcnn import visualize

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python'))
from utils import ShrubConfig, ShrubDataset

Using TensorFlow backend.


## Configurations

In [ ]:
%matplotlib inline 

ROOT = os.path.join(os.path.dirname(os.getcwd()))

# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT, "data/logs")

# Directory with train data
TRAIN_DIR = os.path.join(
    ROOT,
    "data/JuniperMapper", "Photo_Interpretation_Data"
)

# Local path to store the COCO trained weights
COCO_MODEL_PATH = os.path.join(ROOT, "data/logs/mask_rcnn_coco.h5")
# Download COCO trained weights from Releases if needed
if not os.path.exists(COCO_MODEL_PATH):
    utils.download_trained_weights(COCO_MODEL_PATH)

In [ ]:
config = ShrubConfig()
config.display()

## Notebook Preferences

In [3]:
def get_ax(rows=1, cols=1, size=8):
    """Return a Matplotlib Axes array to be used in
    all visualizations in the notebook. Provide a
    central point to control graph sizes.
    
    Change the default size attribute to control the size
    of rendered images
    """
    _, ax = plt.subplots(rows, cols, figsize=(size*cols, size*rows))
    return ax

## Dataset

In [5]:
# Training dataset
dataset_train = ShrubDataset()
dataset_train.load_dataset(
    os.path.join(TRAIN_DIR, "Train", "Images"),
    os.path.join(TRAIN_DIR, "Train", "masks_points.json")
)
dataset_train.prepare()

# Validation dataset
dataset_val = ShrubDataset()
dataset_val.load_dataset(
    os.path.join(TRAIN_DIR, "Test", "Images"),
    os.path.join(TRAIN_DIR, "Test", "masks_points.json")
)
dataset_val.prepare()

In [ ]:
# Load and display random samples
image_ids = np.random.choice(dataset_train.image_ids, 4)
for image_id in image_ids:
    image = dataset_train.load_image(image_id)
    mask, class_ids = dataset_train.load_mask(image_id)
    visualize.display_top_masks(
        image,
        mask,
        class_ids,
        dataset_train.class_names,
        limit=1
    )

## Create Model

In [ ]:
# Create model in training mode
model = modellib.MaskRCNN(
    mode="training",
    config=config,
    model_dir=MODEL_DIR
)

In [ ]:
# Start with weights from COCO
# Also available: imagenet or last
init_with = "coco"

if init_with == "imagenet":
    model.load_weights(model.get_imagenet_weights(), by_name=True)
elif init_with == "coco":
    # Load weights trained on MS COCO, but skip layers that
    # are different due to the different number of classes
    # See README for instructions to download the COCO weights
    excluded_layers = [
        "mrcnn_class_logits",
        "mrcnn_bbox_fc", 
        "mrcnn_bbox",
        "mrcnn_mask"
    ]

    model.load_weights(COCO_MODEL_PATH, by_name=True, exclude=excluded_layers)

elif init_with == "last":
    # Load the last model you trained and continue training
    model.load_weights(model.find_last(), by_name=True)

## Training

Perform transfer learning to fine tunning only the heads. The backbone layers are freezing and training only the randomly initialized layers (i.e. the ones that we didn't use pre-trained weights from MS COCO). To train only the head layers, pass `layers='heads'` to the `train()` function.

In [ ]:
# Train the head branches
# Passing layers="heads" freezes all layers except the head
# layers. You can also pass a regular expression to select
# which layers to train by name pattern.
model.train(
    dataset_train,
    dataset_val, 
    learning_rate=config.LEARNING_RATE, 
    epochs=20, 
    layers='heads'
)


Starting at epoch 0. LR=0.001

Checkpoint Path: C:\Trabajos_CIranzo\Segmentacion\logs\shrubs_cfg\mask_rcnn_shrubs_cfg_{epoch:04d}.h5
Selecting layers to train
fpn_c5p5               (Conv2D)
fpn_c4p4               (Conv2D)
fpn_c3p3               (Conv2D)
fpn_c2p2               (Conv2D)
fpn_p5                 (Conv2D)
fpn_p2                 (Conv2D)
fpn_p3                 (Conv2D)
fpn_p4                 (Conv2D)
In model:  rpn_model
    rpn_conv_shared        (Conv2D)
    rpn_class_raw          (Conv2D)
    rpn_bbox_pred          (Conv2D)
mrcnn_mask_conv1       (TimeDistributed)
mrcnn_mask_bn1         (TimeDistributed)
mrcnn_mask_conv2       (TimeDistributed)
mrcnn_mask_bn2         (TimeDistributed)
mrcnn_class_conv1      (TimeDistributed)
mrcnn_class_bn1        (TimeDistributed)
mrcnn_mask_conv3       (TimeDistributed)
mrcnn_mask_bn3         (TimeDistributed)
mrcnn_class_conv2      (TimeDistributed)
mrcnn_class_bn2        (TimeDistributed)
mrcnn_mask_conv4       (TimeDistributed)
mrcn

c:\Users\Geoforest\miniconda3\envs\mask_rcnn_tf2\lib\site-packages\tensorflow_core\python\framework\indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
c:\Users\Geoforest\miniconda3\envs\mask_rcnn_tf2\lib\site-packages\tensorflow_core\python\framework\indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
c:\Users\Geoforest\miniconda3\envs\mask_rcnn_tf2\lib\site-packages\tensorflow_core\python\framework\indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "


Epoch 1/20
50/50 [==============================] - 800s 16s/step - loss: 1.7676 - val_loss: 1.2656
Epoch 2/20
50/50 [==============================] - 732s 15s/step - loss: 1.1786 - val_loss: 1.7346
Epoch 3/20
50/50 [==============================] - 686s 14s/step - loss: 1.0819 - val_loss: 0.8181
Epoch 4/20
50/50 [==============================] - 684s 14s/step - loss: 1.1191 - val_loss: 1.0832
Epoch 5/20
50/50 [==============================] - 670s 13s/step - loss: 0.9054 - val_loss: 1.6722
Epoch 6/20
50/50 [==============================] - 691s 14s/step - loss: 1.0344 - val_loss: 1.4847
Epoch 7/20
50/50 [==============================] - 679s 14s/step - loss: 1.0921 - val_loss: 1.6569
Epoch 8/20
50/50 [==============================] - 678s 14s/step - loss: 0.9530 - val_loss: 1.9312
Epoch 9/20
50/50 [==============================] - 676s 14s/step - loss: 0.8686 - val_loss: 1.2100
Epoch 10/20
50/50 [==============================] - 683s 14s/step - loss: 0.8259 - val_loss: 1.6712